# Function 6: Black-box optimisation

This notebook documents 13 rounds of model fitting, candidate selection and evaluated outputs for Function 6. Alternative approaches are retained where they informed the weekly decisions.

**Main result:** A noise-aware GP-UCB search reached −0.001592738 in Week 10.

**Usage:** Run the setup and then the weekly sections in order. The initial datasets are in this folder; the full query history is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv). Some early searches generate large candidate grids and may require substantial memory.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.optimize import NonlinearConstraint, differential_evolution
from scipy.spatial import cKDTree
from scipy.stats import norm, truncnorm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF, WhiteKernel


def resolve_initial_path(filename: str) -> Path:
    """Find the initial dataset for Function 6."""
    for folder in (Path.cwd(), Path.cwd() / "function_6"):
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"{filename} not found. Place it in the function_6/ folder."
    )

## Reusable routines

The helper functions below handle standard GP fitting, UCB scoring and recording evaluated queries. The model settings and experiments remain visible within each week. Global/local candidate generation is shared across the early rounds.

In [ ]:
def fit_gp_model(X, y, **settings):
    """Fit a GP using the explicit kernel and settings chosen for a given week."""
    gp = GaussianProcessRegressor(**settings)
    gp.fit(X, y)
    return gp


def ucb_score(mean, std, beta):
    """Upper Confidence Bound: predictive mean plus weighted uncertainty."""
    return mean + beta * std


def add_evaluation(X, y, x_observed, y_observed):
    """Add one completed black-box evaluation to the training history."""
    x_row = np.asarray(x_observed, dtype=float).reshape(1, -1)
    y_value = np.asarray(y_observed, dtype=float).reshape(1)
    if x_row.shape[1] != X.shape[1]:
        raise ValueError("The evaluated input has the wrong number of dimensions.")
    return np.vstack([X, x_row]), np.concatenate([y, y_value])

def sample_hybrid_candidates(best_x, n_global, n_local, seed, *,
                             local_scale=0.08, local_method="truncated"):
    """Combine uniform global samples with local samples near the current best.

    The local method is either "truncated" (truncated normal, inside [0, 1])
    or "clipped" (normal samples clipped to [0, 1]). Random number generation
    follows the same sequence for a given seed as the weekly experiments.
    """
    best_x = np.asarray(best_x, dtype=float).reshape(-1)
    n_features = best_x.size
    rng = np.random.default_rng(seed)

    X_global = rng.uniform(0, 1, size=(n_global, n_features))
    if local_method == "clipped":
        X_local = rng.normal(
            loc=best_x, scale=local_scale, size=(n_local, n_features)
        )
        X_local = np.clip(X_local, 0, 1)
    elif local_method == "truncated":
        X_local = np.empty((n_local, n_features))
        for j in range(n_features):
            a = (0 - best_x[j]) / local_scale
            b = (1 - best_x[j]) / local_scale
            X_local[:, j] = truncnorm.rvs(
                a, b, loc=best_x[j], scale=local_scale,
                size=n_local, random_state=rng
            )
    else:
        raise ValueError("local_method must be 'clipped' or 'truncated'.")

    return np.round(np.vstack([X_global, X_local]), 6)


## Starting data

In [2]:
X = np.load(resolve_initial_path("initial_inputs.npy"))
y = np.load(resolve_initial_path("initial_outputs.npy"))

## Week 1

In [6]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

# UCB exploration parameter
beta = 1.96
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds="fixed"
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True
)
# Generate candidate inputs
# Define the search range for each input variable

x1_values = np.linspace(0, 1, 30)
x2_values = np.linspace(0, 1, 30)
x3_values = np.linspace(0, 1, 30)
x4_values = np.linspace(0, 1, 30)
x5_values = np.linspace(0, 1, 30)

# Generate every combination of x1 and x2
X1_grid, X2_grid, X3_grid, X4_grid, X5_grid = np.meshgrid(
    x1_values,
    x2_values,
    x3_values,
    x4_values,
    x5_values
)

X_grid = np.column_stack([
    X1_grid.ravel(),
    X2_grid.ravel(),
    X3_grid.ravel(),
    X4_grid.ravel(),
    X5_grid.ravel()
])
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_grid,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)
# Select the next query
best_index = np.argmax(UCB)

next_X = X_grid[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.72413793 0.24137931 0.72413793 0.68965517 0.03448276]

Predicted mean:
-0.974398595917828

Predicted standard deviation:
0.3345538663070859

UCB:
-0.3186730179559396


### Evaluated query

**Input:** `(0.724138, 0.241379, 0.724138, 0.689655, 0.034483)`  
**Output:** `-0.7077359838229833`

In [ ]:
X_week1 = np.array([0.724138, 0.241379, 0.724138, 0.689655, 0.034483])
y_week1 = np.array([-0.7077359838229833])
X_new, y_new = add_evaluation(X, y, X_week1, y_week1)

## Week 2

In [10]:
X = X_new
y = y_new

### Increase β to favour exploration

In [11]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 3

### Allow the kernel length scale to vary

In [12]:
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.473)


### Use random candidates instead of a grid

In [13]:
# Generate candidate inputs
n_candidates = 1000000
n_features = 5

np.random.seed(42)

X_candidates = np.random.uniform(
    low=0,
    high=1,
    size=(n_candidates, n_features)
)

X_candidates = np.round(X_candidates, 6)

In [14]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [15]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.243003 0.326254 0.375713 0.744799 0.021132]

Predicted mean:
-0.6892526936530999

Predicted standard deviation:
0.3441649398927907

UCB:
0.34324212602527215


### Evaluated query

**Input:** `(0.243003, 0.326254, 0.375713, 0.744799, 0.021132)`  
**Output:** `-0.6308798453493807`

In [ ]:
X_week2 = np.array([0.243003, 0.326254, 0.375713, 0.744799, 0.021132])
y_week2 = np.array([-0.6308798453493807])
X_new, y_new = add_evaluation(X, y, X_week2, y_week2)

## Week 3

In [19]:
X = X_new
y = y_new

### Option 1: Isotropic GP with global/local sampling · selected

In [20]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 2.5

In [21]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.494)


In [22]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 5
n_global = 850000
n_local = 150000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="truncated",
)

In [23]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [24]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.34122  0.681055 0.004973 0.970296 0.049031]

Predicted mean:
-0.878577141531613

Predicted standard deviation:
0.3599434177405318

UCB:
0.021281402819716466


### Option 2: Anisotropic GP with global/local sampling · alternative

In [25]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1]

beta = 2.5

In [26]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.343, 0.719, 0.663, 0.727, 0.504])


In [27]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 5
n_global = 850000
n_local = 150000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="truncated",
)

In [28]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [29]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.444722 0.123843 0.738463 0.976748 0.194887]

Predicted mean:
-0.38898547387855076

Predicted standard deviation:
0.26885958177160685

UCB:
0.28316348055046636


### Evaluated query

**Input:** `(0.341220, 0.681055, 0.004973, 0.970296, 0.049031)`  
**Output:** `-1.2535352266331634`

In [ ]:
X_week3 = np.array([0.34122 , 0.681055, 0.004973, 0.970296, 0.049031])
y_week3 = np.array([-1.2535352266331634])
X_new, y_new = add_evaluation(X, y, X_week3, y_week3)

## Week 4

In [33]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 1.0 · selected

In [34]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 1.0

In [35]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.475)


In [36]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 5
n_global = 900000
n_local = 100000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [37]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [38]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.42237  0.301682 0.552959 0.793794 0.210044]

Predicted mean:
-0.4665984484404987

Predicted standard deviation:
0.16677117445400433

UCB:
-0.29982727398649434


### Option 2: Anisotropic GP with β = 1.0 · alternative

In [39]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1]

beta = 1.0

In [40]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[2, 0.311, 0.519, 0.506, 0.326])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [41]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 5
n_global = 900000
n_local = 100000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [42]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [43]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[8.95800e-02 1.98973e-01 5.54594e-01 9.69667e-01 2.31000e-04]

Predicted mean:
-0.6222247398641485

Predicted standard deviation:
0.2276017825130534

UCB:
-0.3946229573510951


### Evaluated query

**Input:** `(0.422370, 0.301682, 0.552959, 0.793794, 0.210044)`  
**Output:** `-0.26525159305650087`

In [ ]:
X_week4 = np.array([0.42237, 0.301682, 0.552959, 0.793794, 0.210044])
y_week4 = np.array([-0.26525159305650087])
X_new, y_new = add_evaluation(X, y, X_week4, y_week4)

## Week 5

In [47]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 1.0 · selected

In [48]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 1.0

In [49]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.5)


In [50]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 5
n_global = 900000
n_local = 100000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=47,
    local_scale=0.08,
    local_method="truncated",
)

In [51]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [52]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.383414 0.243802 0.592363 0.971206 0.229882]

Predicted mean:
-0.3180335309199831

Predicted standard deviation:
0.14295032481261324

UCB:
-0.17508320610736985


### Evaluated query

**Input:** `(0.383414, 0.243802, 0.592363, 0.971206, 0.229882)`  
**Output:** `-0.48420647187457855`

In [ ]:
X_week5 = np.array([0.383414, 0.243802, 0.592363, 0.971206, 0.229882])
y_week5 = np.array([-0.48420647187457855])
X_new, y_new = add_evaluation(X, y, X_week5, y_week5)

## Week 6

In [56]:
X = X_new
y = y_new

### Matérn 3/2 GP with β = 1

In [57]:
# Function 6 - Week 6
noise_assumption = 1e-10
beta = 1.0

kernel = Matern(
    length_scale=0.5,
    length_scale_bounds=(0.05, 2.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Optimised kernel:", model.kernel_)

Optimised kernel: Matern(length_scale=0.783, nu=1.5)


In [58]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 5
n_global = 900000
n_local = 100000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=48,
    local_scale=0.08,
    local_method="truncated",
)

In [59]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [60]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.42867  0.257351 0.579181 0.753304 0.127558]

Predicted mean:
-0.2940202260228595

Predicted standard deviation:
0.08403821264824317

UCB:
-0.2099820133746163


### Evaluated query

**Input:** `(0.428670, 0.257351, 0.579181, 0.753304, 0.127558)`  
**Output:** `-0.15087634205973416`

In [ ]:
X_week6 = np.array([0.42867 , 0.257351, 0.579181, 0.753304, 0.127558])
y_week6 = np.array([-0.15087634205973416])
X_new, y_new = add_evaluation(X, y, X_week6, y_week6)

## Week 7

In [64]:
# All 26 observations, including Week 6
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (26, 5)
assert y.shape == (26,)

# Model A: anisotropic Matérn + amplitude
kernel_ard = ConstantKernel(
    1.0, (0.03, 100.0)
) * Matern(
    length_scale=np.full(5, 0.5),
    length_scale_bounds=(0.02, 4.0),
    nu=1.5
)

# Model B: isotropic Matérn + amplitude
kernel_iso = ConstantKernel(
    1.0, (0.03, 100.0)
) * Matern(
    length_scale=0.5,
    length_scale_bounds=(0.02, 4.0),
    nu=1.5
)

def fit_gp(kernel):
    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )
    return model.fit(X, y)

gp_ard = fit_gp(kernel_ard)
gp_iso = fit_gp(kernel_iso)

print("Anisotropic:", gp_ard.kernel_)
print("Isotropic:", gp_iso.kernel_)

# Combine GP predictions and account for
# disagreement between the two models
def ensemble_predict(points):
    mu_a, std_a = gp_ard.predict(
        points, return_std=True
    )
    mu_i, std_i = gp_iso.predict(
        points, return_std=True
    )

    mu = 0.5 * (mu_a + mu_i)

    variance = 0.5 * (
        std_a**2 + (mu_a - mu)**2
    ) + 0.5 * (
        std_i**2 + (mu_i - mu)**2
    )

    return mu, np.sqrt(variance)

beta = 0.5
min_distance = 0.05
tree = cKDTree(X)

# Vectorised acquisition objective
def negative_ucb(points):
    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = ensemble_predict(candidates)
    distances = tree.query(candidates)[0]

    penalty = 10 * np.maximum(
        0, min_distance - distances
    )

    scores = -(ucb_score(mu, std, beta)) + penalty

    return scores if points.ndim == 2 else scores[0]

# Optimise from several seeds
solutions = []

for seed in [42, 48, 59]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 5,
        seed=seed,
        popsize=14,
        maxiter=180,
        tol=1e-6,
        updating="deferred",
        vectorized=True,
        polish=False
    )
    solutions.append(result)

best_result = min(solutions, key=lambda r: r.fun)
next_X = np.round(best_result.x, 6)

mu, std = ensemble_predict(
    next_X.reshape(1, -1)
)

print("Week 7 query:", next_X)
print("Predicted mean:", mu[0])
print("Model-averaged std:", std[0])
print("Acquisition score:", mu[0] + beta * std[0])
print("Nearest distance:", tree.query(next_X)[0])

Anisotropic: 1.81**2 * Matern(length_scale=[0.943, 2.27, 2.19, 1.53, 2.43], nu=1.5)
Isotropic: 1.25**2 * Matern(length_scale=0.998, nu=1.5)
Week 7 query: [0.434516 0.153027 0.655572 0.73073  0.      ]
Predicted mean: -0.13895602297516763
Model-averaged std: 0.1286208971271951
Acquisition score: -0.07464557441157008
Nearest distance: 0.1831225994054257


### Evaluated query

**Input:** `(0.434516, 0.153027, 0.655572, 0.730730, 0.000000)`  
**Output:** `-0.38414314030475416`

In [ ]:
X_week7 = np.array([0.434516, 0.153027, 0.655572, 0.73073 , 0.])
y_week7 = np.array([-0.38414314030475416])
X_new, y_new = add_evaluation(X, y, X_week7, y_week7)

## Week 8

In [68]:
# All observations through Week 7
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (27, 5)
assert y.shape == (27,)

# Selected GP model
kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(0.03, 100.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 4.0),
        nu=1.5
    )
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Optimised kernel:", model.kernel_)

# Acquisition settings
beta = 1.0
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(points):
    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = model.predict(
        candidates,
        return_std=True
    )

    distances = tree.query(candidates)[0]

    penalty = 1000 * np.maximum(
        0, min_distance - distances
    )

    scores = -(ucb_score(mu, std, beta)) + penalty

    return scores if points.ndim == 2 else scores[0]

# Search the full five-dimensional domain
solutions = []

for seed in [42, 48, 59]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 5,
        seed=seed,
        popsize=15,
        maxiter=180,
        tol=1e-7,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    if tree.query(result.x)[0] >= min_distance - 1e-5:
        solutions.append(result)

if not solutions:
    raise RuntimeError("No feasible candidate found.")

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)

mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Week 8 query:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print("Nearest distance:", tree.query(next_X)[0])

Optimised kernel: 1.15**2 * Matern(length_scale=0.91, nu=1.5)
Week 8 query: [0.449057 0.23196  0.549734 0.701289 0.139757]
Predicted mean: -0.1738122179365671
Predicted std: 0.060633816031698234
UCB: -0.11317840190486886
Nearest distance: 0.06915080827437949


### Evaluated query

**Input:** `(0.449057, 0.231960, 0.549734, 0.701289, 0.139757)`  
**Output:** `-0.2997280518488676`

In [ ]:
X_week8 = np.array([0.449057, 0.23196 , 0.549734, 0.701289, 0.139757])
y_week8 = np.array([-0.2997280518488676])
X_new, y_new = add_evaluation(X, y, X_week8, y_week8)

## Week 9

In [72]:
# Observations through Week 8
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (28, 5)
assert y.shape == (28,)
# Selected GP model
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 100.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 4.0),
        nu=1.5
    )
    + WhiteKernel(
        noise_level=0.01,
        noise_level_bounds=(1e-6, 0.4)
    )
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Optimised kernel:", model.kernel_)
# Latent-function prediction
# Subtract the fitted observation-noise
# variance before calculating GP-UCB.

noise_variance = (
    model.kernel_.k2.noise_level * np.var(y)
)

def latent_predict(points):
    mu, total_std = model.predict(
        points,
        return_std=True
    )

    latent_std = np.sqrt(
        np.maximum(
            total_std**2 - noise_variance,
            0.0
        )
    )

    return mu, latent_std
# Acquisition settings
beta = 1.0
min_distance = 0.10
tree = cKDTree(X)

def negative_ucb(points):
    points = np.asarray(points)

    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = latent_predict(candidates)
    distances = tree.query(candidates)[0]

    ucb = ucb_score(mu, std, beta)

    penalty = 1000 * np.maximum(
        0,
        min_distance - distances
    )

    scores = -ucb + penalty

    return (
        scores if points.ndim == 2
        else scores[0]
    )
# Optimise GP-UCB
solutions = []

for seed in [42, 48, 59]:

    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 5,
        seed=seed,
        popsize=14,
        maxiter=150,
        tol=1e-8,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    if tree.query(result.x)[0] >= (
        min_distance - 1e-4
    ):
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible candidate was found."
    )

best_result = min(
    solutions,
    key=lambda r: r.fun
)

next_X = np.round(best_result.x, 6)
# Report recommendation
mu, std = latent_predict(
    next_X.reshape(1, -1)
)

print("Week 9 query:", next_X)
print("Predicted mean:", mu[0])
print("Latent predicted std:", std[0])
print("GP-UCB:", mu[0] + beta * std[0])
print(
    "Nearest distance:",
    tree.query(next_X)[0]
)
print("Current best:", y.max())

Optimised kernel: 1.43**2 * Matern(length_scale=1.31, nu=1.5) + WhiteKernel(noise_level=0.00888)
Week 9 query: [0.41338  0.33591  0.636191 0.758932 0.109869]
Predicted mean: -0.25550889490610673
Latent predicted std: 0.083487175545223
GP-UCB: -0.17202171936088373
Nearest distance: 0.10000007892996886
Current best: -0.15087634205973416


### Evaluated query

**Input:** `(0.413380, 0.335910, 0.636191, 0.758932, 0.109869)`  
**Output:** `-0.08681640739317552`

In [ ]:
X_week9 = np.array([0.41338 , 0.33591 , 0.636191, 0.758932, 0.109869])
y_week9 = np.array([-0.08681640739317552])
X_new, y_new = add_evaluation(X, y, X_week9, y_week9)

## Week 10

In [76]:
# Observations through Week 9
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (29, 5)
assert y.shape == (29,)
# Fit selected GP
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(0.03, 100.0)
    )
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 4.0),
        nu=1.5
    )
    + WhiteKernel(
        noise_level=0.01,
        noise_level_bounds=(1e-6, 0.4)
    )
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)
print("Optimised kernel:", model.kernel_)
# Predict latent function uncertainty
noise_variance = (
    model.kernel_.k2.noise_level
    * np.var(y)
)

def latent_predict(points):
    mu, total_std = model.predict(
        points,
        return_std=True
    )

    latent_std = np.sqrt(
        np.maximum(
            total_std**2 - noise_variance,
            0.0
        )
    )

    return mu, latent_std
# GP-UCB acquisition settings
beta = 1.0
min_distance = 0.06

tree = cKDTree(X)

def negative_ucb(x):
    x = np.asarray(x).reshape(1, -1)

    mu, std = latent_predict(x)

    return -float(
        mu[0] + beta * std[0]
    )

distance_constraint = NonlinearConstraint(
    lambda x: tree.query(x)[0],
    min_distance + 2e-5,
    np.inf
)
# Optimise acquisition function
solutions = []

for seed in [42, 59, 111]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 5,
        constraints=distance_constraint,
        seed=seed,
        popsize=14,
        maxiter=145,
        tol=1e-7,
        polish=True
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible acquisition candidate found."
    )

best_result = min(
    solutions,
    key=lambda r: r.fun
)

next_X = np.round(best_result.x, 6)
# Evaluate recommendation
mu, std = latent_predict(
    next_X.reshape(1, -1)
)

best_y = np.max(y)

prob_improvement = norm.cdf(
    (mu[0] - best_y) / std[0]
)

print("Week 10 query:", next_X)
print("Predicted mean:", mu[0])
print("Latent std:", std[0])
print("GP-UCB:", mu[0] + beta * std[0])
print(
    "Probability of improvement:",
    prob_improvement
)
print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)
print("Current best output:", best_y)

Optimised kernel: 1.31**2 * Matern(length_scale=1.15, nu=1.5) + WhiteKernel(noise_level=0.00453)


/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_differentiable_functions.py:728: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(delta_x, delta_g)
/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_differentiable_functions.py:728: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(delta_x, delta_g)
/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_differentiable_functions.py:728: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.updat

Week 10 query: [0.391479 0.366644 0.681208 0.766389 0.083006]
Predicted mean: -0.12774117430233045
Latent std: 0.06627617194651238
GP-UCB: -0.061465002355818074
Probability of improvement: 0.26845631740455755
Nearest observation distance: 0.06502299950017693
Current best output: -0.08681640739317552


### Evaluated query

**Input:** `(0.391479, 0.366644, 0.681208, 0.766389, 0.083006)`  
**Output:** `-0.0015927376191079978`

In [ ]:
X_week10 = np.array([0.391479, 0.366644, 0.681208, 0.766389, 0.083006])
y_week10 = np.array([-0.0015927376191079978])
X_new, y_new = add_evaluation(X, y, X_week10, y_week10)

## Week 11

In [79]:
# Observations through Week 10
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (30, 5)
assert y.shape == (30,)

best_idx = np.argmax(y)

best_X = X[best_idx]
best_y = y[best_idx]

print("Best observed input:", best_X)
print("Best observed output:", best_y)
# Fit two complementary GP models
kernel_ard_white = (
    ConstantKernel(1.0, (0.03, 100.0))
    * Matern(
        length_scale=np.full(5, 0.5),
        length_scale_bounds=(0.02, 5.0),
        nu=1.5
    )
    + WhiteKernel(
        noise_level=0.01,
        noise_level_bounds=(1e-6, 0.4)
    )
)

kernel_iso = (
    ConstantKernel(1.0, (0.03, 100.0))
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.02, 5.0),
        nu=1.5
    )
)

def fit_gp(kernel):
    return GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=4,
        random_state=42
    ).fit(X, y)

gp_ard_white = fit_gp(kernel_ard_white)
gp_iso = fit_gp(kernel_iso)

print("\nAnisotropic regularised GP:")
print(gp_ard_white.kernel_)

print("\nIsotropic interpolating GP:")
print(gp_iso.kernel_)
# Follow the Week 9 to Week 10 direction
previous_X = X[-2]
latest_X = X[-1]

direction = latest_X - previous_X

step_fraction = 0.5

next_X = np.clip(
    latest_X + step_fraction * direction,
    0, 1
)

next_X = np.round(next_X, 6)
# Check distance from previous queries
tree = cKDTree(X)

nearest_distance = tree.query(next_X)[0]

min_distance = 0.03

assert nearest_distance >= min_distance
# Predict candidate performance
mu_iso, std_iso = gp_iso.predict(
    next_X.reshape(1, -1),
    return_std=True
)

mu_ard, total_std_ard = gp_ard_white.predict(
    next_X.reshape(1, -1),
    return_std=True
)

# Remove the WhiteKernel observation-noise
# contribution to estimate latent uncertainty.

noise_variance = (
    gp_ard_white.kernel_.k2.noise_level
    * gp_ard_white._y_train_std**2
)

latent_std_ard = np.sqrt(
    np.maximum(
        total_std_ard[0]**2 - noise_variance,
        0
    )
)
# Report Week 11 candidate
print("\nWeek 11 proposed input:")
print(next_X)

print("\nIsotropic GP:")
print("Predicted mean:", mu_iso[0])
print("Predicted std:", std_iso[0])

print("\nAnisotropic regularised GP:")
print("Predicted mean:", mu_ard[0])
print("Latent predicted std:", latent_std_ard)

print("\nNearest observation distance:",
      nearest_distance)

print("Current best output:", best_y)

Best observed input: [0.391479 0.366644 0.681208 0.766389 0.083006]
Best observed output: -0.0015927376191079978

Anisotropic regularised GP:
1.92**2 * Matern(length_scale=[1.19, 1.49, 3.14, 1.94, 3.17], nu=1.5) + WhiteKernel(noise_level=0.00582)

Isotropic interpolating GP:
1.05**2 * Matern(length_scale=0.79, nu=1.5)

Week 11 proposed input:
[0.380529 0.382011 0.703717 0.770118 0.069574]

Isotropic GP:
Predicted mean: 0.020685933955634228
Predicted std: 0.022548073343143942

Anisotropic regularised GP:
Predicted mean: -0.03289515962744183
Latent predicted std: 0.048544664016131084

Nearest observation distance: 0.03251194142157615
Current best output: -0.0015927376191079978


### Evaluated query

**Input:** `(0.380529, 0.382011, 0.703717, 0.770118, 0.069574)`  
**Output:** `-0.2319206310109243`

In [ ]:
X_week11 = np.array([0.380529, 0.382011, 0.703717, 0.770118, 0.069574])
y_week11 = np.array([-0.2319206310109243])
X_new, y_new = add_evaluation(X, y, X_week11, y_week11)

## Week 12

In [82]:
# Observations through Week 11
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (31, 5)
assert y.shape == (31,)

best_idx = np.argmax(y)
best_X = X[best_idx]
best_y = y[best_idx]

print("Current best input:", best_X)
print("Current best output:", best_y)
# Refit two complementary GP models
gp_interpolating = GaussianProcessRegressor(
    kernel=(
        ConstantKernel(1.0, (0.03, 100.0))
        * Matern(
            length_scale=0.5,
            length_scale_bounds=(0.02, 5.0),
            nu=1.5
        )
    ),
    alpha=1e-8,
    normalize_y=True,
    n_restarts_optimizer=2,
    random_state=42
)

gp_regularised = GaussianProcessRegressor(
    kernel=(
        ConstantKernel(1.0, (0.03, 100.0))
        * Matern(
            length_scale=np.full(5, 0.5),
            length_scale_bounds=(0.02, 5.0),
            nu=1.5
        )
        + WhiteKernel(
            noise_level=0.01,
            noise_level_bounds=(1e-5, 0.4)
        )
    ),
    alpha=1e-8,
    normalize_y=True,
    n_restarts_optimizer=2,
    random_state=42
)

gp_interpolating.fit(X, y)
gp_regularised.fit(X, y)
# Search between Weeks 9 and 10
week9_X = X[-3]
week10_X = X[-2]

t_values = np.linspace(0.5, 0.9, 81)

candidates = np.array([
    week9_X + t * (week10_X - week9_X)
    for t in t_values
])

means, stds = gp_interpolating.predict(
    candidates,
    return_std=True
)
# Apply a modest distance constraint
tree = cKDTree(X)
distances = tree.query(candidates)[0]

min_distance = 0.015

scores = means.copy()
scores[distances < min_distance] = -np.inf

best_candidate_idx = np.argmax(scores)

print("Best GP position:",
      t_values[best_candidate_idx])
# Use a controlled t = 0.70 candidate
t_selected = 0.70

next_X = (
    week9_X
    + t_selected * (week10_X - week9_X)
)

next_X = np.round(next_X, 6)
# Compare predictions
print("\nWeek 12 proposed input:")
print(" ".join(f"{v:.6f}" for v in next_X))

for name, model in [
    ("Interpolating GP", gp_interpolating),
    ("Regularised GP", gp_regularised)
]:
    mu, std = model.predict(
        next_X.reshape(1, -1),
        return_std=True
    )

    print(f"\n{name}:")
    print("Kernel:", model.kernel_)
    print("Predicted mean:", mu[0])
    print("Predicted std:", std[0])

print("\nNearest observation distance:",
      tree.query(next_X)[0])

print("Current best output:", best_y)

Current best input: [0.391479 0.366644 0.681208 0.766389 0.083006]
Current best output: -0.0015927376191079978
Best GP position: 0.735

Week 12 proposed input:
0.398049 0.357424 0.667703 0.764152 0.091065

Interpolating GP:
Kernel: 0.843**2 * Matern(length_scale=0.274, nu=1.5)
Predicted mean: 0.03226803116095067
Predicted std: 0.023943008644288808

Regularised GP:
Kernel: 2.12**2 * Matern(length_scale=[1.4, 1.85, 3.66, 2.25, 3.67], nu=1.5) + WhiteKernel(noise_level=0.0153)
Predicted mean: -0.13290730582154797
Predicted std: 0.09221026043179932

Nearest observation distance: 0.019506664886648335
Current best output: -0.0015927376191079978


### Evaluated query

**Input:** `(0.398049, 0.357424, 0.667703, 0.764152, 0.091065)`  
**Output:** `-0.09724916261507487`

In [ ]:
X_week12 = np.array([0.398049, 0.357424, 0.667703, 0.764152, 0.091065])
y_week12 = np.array([-0.09724916261507487])
X_new, y_new = add_evaluation(X, y, X_week12, y_week12)

## Week 13

In [85]:
# Observations through Week 12
X_all = np.asarray(X_new, dtype=float)
y_all = np.asarray(y_new, dtype=float).ravel()

assert X_all.shape == (32, 5)
assert y_all.shape == (32,)

best_idx = np.argmax(y_all)

print("Current best input:", X_all[best_idx])
print("Current best output:", y_all[best_idx])
# Identify recent observations
week9_X = X_all[-4]
week10_X = X_all[-3]
week11_X = X_all[-2]
week12_X = X_all[-1]
# Small extrapolation beyond Week 10
t_selected = 1.03

next_X = (
    week9_X
    + t_selected * (week10_X - week9_X)
)

next_X = np.round(next_X, 6)

print("\nFinal Week 13 input:")
print(" ".join(f"{v:.6f}" for v in next_X))

print(
    "Distance from Week 10:",
    np.linalg.norm(next_X - week10_X)
)
# Fit complementary GP models
def fit_gp(target, kernel):
    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-7,
        normalize_y=True,
        n_restarts_optimizer=2,
        random_state=42
    )

    model.fit(X_all, target)
    return model

raw_gp = fit_gp(
    y_all,
    RBF(
        length_scale=0.5,
        length_scale_bounds=(0.015, 10)
    )
)

assert np.all(y_all < 0)

sqrt_gp = fit_gp(
    np.sqrt(-y_all),
    ConstantKernel(1.0, (0.01, 100))
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.015, 10),
        nu=2.5
    )
)

log_gp = fit_gp(
    np.log(-y_all),
    ConstantKernel(1.0, (0.01, 100))
    * Matern(
        length_scale=0.5,
        length_scale_bounds=(0.015, 10),
        nu=1.5
    )
)
# Evaluate candidate
point = next_X.reshape(1, -1)

raw_mu = raw_gp.predict(point)[0]

sqrt_mu, sqrt_std = sqrt_gp.predict(
    point, return_std=True
)

log_mu, log_std = log_gp.predict(
    point, return_std=True
)

# Back-transform expected values
sqrt_prediction = -(
    sqrt_mu[0]**2 + sqrt_std[0]**2
)

log_prediction = -np.exp(
    log_mu[0] + 0.5 * log_std[0]**2
)

print("\nModel predictions:")
print(f"Raw RBF GP:       {raw_mu:.8f}")
print(f"Square-root GP:   {sqrt_prediction:.8f}")
print(f"Log GP:           {log_prediction:.8f}")

print("\nCurrent best:", y_all.max())
print("\nFINAL SUBMISSION:")
print(" ".join(f"{v:.6f}" for v in next_X))

Current best input: [0.391479 0.366644 0.681208 0.766389 0.083006]
Current best output: -0.0015927376191079978

Final Week 13 input:
0.390822 0.367566 0.682559 0.766613 0.082200
Distance from Week 10: 0.0019510884141934633

Model predictions:
Raw RBF GP:       0.00164956
Square-root GP:   -0.00127958
Log GP:           -0.00180591

Current best: -0.0015927376191079978

FINAL SUBMISSION:
0.390822 0.367566 0.682559 0.766613 0.082200


### Week 13 evaluated result

**Input:** `(0.390822, 0.367566, 0.682559, 0.766613, 0.082200)`  
**Output:** `-0.13267331370444363`

The query-selection calculations above use results available through Week 12. The full record is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv).